# NYC EMS Bronze Data Ingestion

## Purpose

This notebook ingests NYC EMS Incident Dispatch CSV files from
2019 to 2025 into the Bronze layer.

## Responsibilities

- Read all 47 source CSV files
- Apply an explicit 31-column schema
- Preserve source values without business transformations
- Add ingestion metadata
- Validate row counts by year
- Validate required incident identifiers
- Write the validated data to a partitioned Bronze Delta table

## Source

- NYC Open Data
- Dataset: EMS Incident Dispatch Data
- Dataset ID: `76xm-jjuj`
- Reporting period: 2019–2025
- Expected rows: 10,881,496

In [16]:
# Import PySpark components
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 18, Finished, Available, Finished, False)

In [17]:
# Define schema of 31 columns
# Read all columns as String, maintain the raw data

source_columns = [
    "incident_id",
    "incident_datetime",
    "initial_call_type",
    "initial_severity_level_code",
    "final_call_type",
    "final_severity_level_code",
    "first_assignment_datetime",
    "valid_dispatch_rspns_time_indc",
    "dispatch_response_seconds_qy",
    "first_activation_datetime",
    "first_on_scene_datetime",
    "valid_incident_rspns_time_indc",
    "incident_response_seconds_qy",
    "incident_travel_tm_seconds_qy",
    "first_to_hosp_datetime",
    "first_hosp_arrival_datetime",
    "incident_close_datetime",
    "held_indicator",
    "incident_disposition_code",
    "borough",
    "incident_dispatch_area",
    "zipcode",
    "policeprecinct",
    "citycouncildistrict",
    "communitydistrict",
    "communityschooldistrict",
    "congressionaldistrict",
    "reopen_indicator",
    "special_event_indicator",
    "standby_indicator",
    "transfer_indicator"
]

source_schema = StructType([
    StructField(column_name, StringType(), True)
    for column_name in source_columns
])

print(f"Expected source columns: {len(source_columns)}")

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 19, Finished, Available, Finished, False)

Expected source columns: 31


In [18]:
# Check csv files number
expected_file_counts = {
    2025: 7,
    2024: 7,
    2023: 7,
    2022: 7,
    2021: 6,
    2020: 6,
    2019: 7
}

file_validation_results = []
total_csv_files = 0

for year, expected_file_count in expected_file_counts.items():
    year_path = f"Files/raw/ems_incidents/year={year}"

    files = notebookutils.fs.ls(year_path)

    csv_files = [
        file
        for file in files
        if file.name.lower().endswith(".csv")
    ]

    actual_file_count = len(csv_files)
    total_csv_files += actual_file_count

    file_validation_results.append((
        year,
        expected_file_count,
        actual_file_count,
        "PASS"
        if expected_file_count == actual_file_count
        else "FAIL"
    ))

df_file_validation = spark.createDataFrame(
    file_validation_results,
    [
        "year",
        "expected_files",
        "actual_files",
        "status"
    ]
)

display(
    df_file_validation.orderBy(
        F.col("year").desc()
    )
)

assert total_csv_files == 47, (
    f"Expected 47 CSV files, found {total_csv_files}"
)

assert all(
    result[3] == "PASS"
    for result in file_validation_results
), "One or more yearly file-count checks failed."

print("Source file validation passed.")
print(f"Total CSV files: {total_csv_files}")

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2c39850b-d5a7-4ca4-985f-e42638cd2db1)

Source file validation passed.
Total CSV files: 47


In [19]:
# Read all csv files
raw_data_path = "Files/raw/ems_incidents/year=*/*.csv"

df_bronze_source = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .option("enforceSchema", "false")
    .schema(source_schema)
    .load(raw_data_path)
    .withColumn(
        "_source_file",
        F.input_file_name()
    )
    .withColumn(
        "_source_year",
        F.regexp_extract(
            F.col("_source_file"),
            r"year=(\d{4})",
            1
        ).cast("int")
    )
    .withColumn(
        "_ingested_at",
        F.current_timestamp()
    )
)

print("CSV files were successfully registered as a DataFrame.")

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 21, Finished, Available, Finished, False)

CSV files were successfully registered as a DataFrame.


In [20]:
# Chech schema and samples
df_bronze_source.printSchema()

display(
    df_bronze_source.select(
        "incident_id",
        "incident_datetime",
        "initial_call_type",
        "incident_response_seconds_qy",
        "borough",
        "_source_year",
        "_source_file"
    ).limit(10)
)

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 22, Finished, Available, Finished, False)

root
 |-- incident_id: string (nullable = true)
 |-- incident_datetime: string (nullable = true)
 |-- initial_call_type: string (nullable = true)
 |-- initial_severity_level_code: string (nullable = true)
 |-- final_call_type: string (nullable = true)
 |-- final_severity_level_code: string (nullable = true)
 |-- first_assignment_datetime: string (nullable = true)
 |-- valid_dispatch_rspns_time_indc: string (nullable = true)
 |-- dispatch_response_seconds_qy: string (nullable = true)
 |-- first_activation_datetime: string (nullable = true)
 |-- first_on_scene_datetime: string (nullable = true)
 |-- valid_incident_rspns_time_indc: string (nullable = true)
 |-- incident_response_seconds_qy: string (nullable = true)
 |-- incident_travel_tm_seconds_qy: string (nullable = true)
 |-- first_to_hosp_datetime: string (nullable = true)
 |-- first_hosp_arrival_datetime: string (nullable = true)
 |-- incident_close_datetime: string (nullable = true)
 |-- held_indicator: string (nullable = true)
 |-

SynapseWidget(Synapse.DataFrame, d59f8115-12bd-4842-bba6-efa6ce9179d6)

In [21]:
# Verify rows of each year
expected_counts = {
    2025: 1612273,
    2024: 1630447,
    2023: 1617839,
    2022: 1583531,
    2021: 1491454,
    2020: 1412701,
    2019: 1533251
}

year_validation_df = (
    df_bronze_source
    .groupBy("_source_year")
    .agg(
        F.count("*").alias("actual_rows"),
        F.sum(
            F.when(
                F.col("incident_id").isNull() |
                (F.trim(F.col("incident_id")) == ""),
                1
            ).otherwise(0)
        ).alias("missing_incident_ids")
    )
    .orderBy(F.col("_source_year").desc())
)

validation_rows = year_validation_df.collect()

validation_results = []

for row in validation_rows:
    year = row["_source_year"]
    actual_rows = row["actual_rows"]
    expected_rows = expected_counts.get(year)
    missing_ids = row["missing_incident_ids"]

    validation_results.append((
        year,
        expected_rows,
        actual_rows,
        actual_rows - expected_rows if expected_rows is not None else None,
        missing_ids,
        "PASS"
        if expected_rows == actual_rows and missing_ids == 0
        else "FAIL"
    ))

validation_schema = [
    "year",
    "expected_rows",
    "row_difference",
    "missing_incident_ids",
    "status"
]

df_validation_results = spark.createDataFrame(
    validation_results,
    validation_schema
)

display(
    df_validation_results.orderBy(
        F.col("year").desc()
    )
)

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2c498714-73de-4e6b-b804-42df412adad4)

In [22]:
# Final assertion
actual_counts = {
    row["_source_year"]: row["actual_rows"]
    for row in validation_rows
}

total_actual_rows = sum(actual_counts.values())
total_expected_rows = sum(expected_counts.values())

failed_years = [
    row
    for row in validation_results
    if row[-1] != "PASS"
]

assert actual_counts == expected_counts, (
    f"Year row-count validation failed: {actual_counts}"
)

assert len(failed_years) == 0, (
    f"Data-quality validation failed: {failed_years}"
)

assert total_actual_rows == 10881496, (
    f"Expected 10,881,496 rows but found {total_actual_rows}"
)

print("Bronze source validation passed.")
print(f"Total expected rows: {total_expected_rows:,}")
print(f"Total actual rows:   {total_actual_rows:,}")
print("Missing incident IDs: 0")
print("Validated years: 2019–2025")

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 24, Finished, Available, Finished, False)

Bronze source validation passed.
Total expected rows: 10,881,496
Total actual rows:   10,881,496
Missing incident IDs: 0
Validated years: 2019–2025


## Write Bronze Delta Table

The validated source records are written to a Delta table without
business-level cleaning or type conversion.

The table is partitioned by source year to improve year-based filtering
and future incremental processing.

In [23]:
bronze_table_name = "bronze_ems_incidents"

(
    df_bronze_source.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_source_year")
    .saveAsTable(bronze_table_name)
)

print(f"Delta table created: {bronze_table_name}")

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 25, Finished, Available, Finished, False)

Delta table created: bronze_ems_incidents


In [24]:
# Refresh and check tale
spark.catalog.refreshTable(bronze_table_name)

df_bronze_delta = spark.table(bronze_table_name)

print(f"Bronze Delta columns: {len(df_bronze_delta.columns)}")
df_bronze_delta.printSchema()

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 26, Finished, Available, Finished, False)

Bronze Delta columns: 34
root
 |-- incident_id: string (nullable = true)
 |-- incident_datetime: string (nullable = true)
 |-- initial_call_type: string (nullable = true)
 |-- initial_severity_level_code: string (nullable = true)
 |-- final_call_type: string (nullable = true)
 |-- final_severity_level_code: string (nullable = true)
 |-- first_assignment_datetime: string (nullable = true)
 |-- valid_dispatch_rspns_time_indc: string (nullable = true)
 |-- dispatch_response_seconds_qy: string (nullable = true)
 |-- first_activation_datetime: string (nullable = true)
 |-- first_on_scene_datetime: string (nullable = true)
 |-- valid_incident_rspns_time_indc: string (nullable = true)
 |-- incident_response_seconds_qy: string (nullable = true)
 |-- incident_travel_tm_seconds_qy: string (nullable = true)
 |-- first_to_hosp_datetime: string (nullable = true)
 |-- first_hosp_arrival_datetime: string (nullable = true)
 |-- incident_close_datetime: string (nullable = true)
 |-- held_indicator: str

In [25]:
# Verify rows of each year of Delta table
delta_year_counts_df = (
    df_bronze_delta
    .groupBy("_source_year")
    .agg(
        F.count("*").alias("delta_rows"),
        F.sum(
            F.when(
                F.col("incident_id").isNull() |
                (F.trim(F.col("incident_id")) == ""),
                1
            ).otherwise(0)
        ).alias("missing_incident_ids")
    )
    .orderBy(F.col("_source_year").desc())
)

display(delta_year_counts_df)

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cbb654bc-ed9d-4c5f-8b78-332c681cc828)

In [26]:
# Automatically assert Delta results

delta_count_rows = delta_year_counts_df.collect()

delta_counts = {
    row["_source_year"]: row["delta_rows"]
    for row in delta_count_rows
}

delta_missing_ids = sum(
    row["missing_incident_ids"]
    for row in delta_count_rows
)

delta_total_rows = sum(delta_counts.values())

assert delta_counts == expected_counts, (
    f"Delta year counts do not match expected counts: {delta_counts}"
)

assert delta_total_rows == 10881496, (
    f"Expected 10,881,496 Delta rows, found {delta_total_rows}"
)

assert delta_missing_ids == 0, (
    f"Found {delta_missing_ids} missing incident IDs"
)

print("Bronze Delta validation passed.")
print(f"Delta table: {bronze_table_name}")
print(f"Total rows: {delta_total_rows:,}")
print(f"Source columns: {len(source_columns)}")
print(f"Total table columns: {len(df_bronze_delta.columns)}")
print("Partition column: _source_year")

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 28, Finished, Available, Finished, False)

Bronze Delta validation passed.
Delta table: bronze_ems_incidents
Total rows: 10,881,496
Source columns: 31
Total table columns: 34
Partition column: _source_year


## Bronze Ingestion Audit

The ingestion audit table records expected and actual row counts,
missing identifier counts and validation results for each source year.

In [27]:
# Create Bronze audit table
df_bronze_audit = (
    df_validation_results
    .withColumn(
        "source_path",
        F.lit(raw_data_path)
    )
    .withColumn(
        "bronze_table",
        F.lit(bronze_table_name)
    )
    .withColumn(
        "validated_at",
        F.current_timestamp()
    )
)

(
    df_bronze_audit.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_ems_ingestion_audit")
)

display(
    spark.table("bronze_ems_ingestion_audit")
    .orderBy(F.col("year").desc())
)

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 22680d4f-2b5c-4c2c-9d5d-4a6fdb4cb230)

In [28]:
# Check Delta table detail
display(
    spark.sql("""
        DESCRIBE DETAIL bronze_ems_incidents
    """)
)

StatementMeta(, 000b1631-d526-428b-91c3-c34220135d2d, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 25133ca9-dc08-4164-94e0-ea15e7bd1123)

## Bronze Layer Result

The Bronze ingestion completed successfully.

### Output tables

- `bronze_ems_incidents`
- `bronze_ems_ingestion_audit`

### Validation result

- 47 source CSV files validated
- 10,881,496 source records validated
- 31 original source columns validated
- 7 source years identified
- Delta table partitioned by `_source_year`
- 0 missing incident identifiers
- All yearly file-count and row-count validations passed

Business-level cleaning, deduplication and type conversion are performed
in the Silver layer.